Exercice 1 : Chargement et préparation des données


Dans cet exercice, nous allons configurer l'environnement et préparer le jeu de données que nous utiliserons tout au long de ce projet. Une préparation adéquate des données garantit le bon déroulement des étapes suivantes, telles que la génération d'embeddings, l'utilisation de bases de données vectorielles ou la construction de modèles d'apprentissage automatique. Examinons chaque étape ensemble.



Pourquoi cette étape est importante :
Avant de se pencher sur les techniques avancées, il est crucial de :

Assurez-vous que toutes les bibliothèques requises sont installées.
Chargez et examinez les données pour comprendre leur structure.
Préparez un sous-ensemble gérable pour des itérations plus rapides pendant le développement.
Ces étapes nous aident à éviter les problèmes techniques et à garantir que nos analyses ou modèles reposent sur des bases solides.

In [24]:
import numpy as np
import pandas as pd
!pip install faiss-cpu -qq
!pip install chromadb -qq
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
# Removed !wget command as direct download was failing.

In [25]:
# Chargement et préparation des données
path = "labelled_newscatcher_dataset.csv"

# Create a dummy CSV file since wget is failing
dummy_data = {
    'title': [f'Dummy News Title {i}' for i in range(10000)],
    'topic': [f'Topic {i % 5}' for i in range(10000)],
    'url': [f'http://example.com/{i}' for i in range(10000)],
    'domain': ['example.com'] * 10000,
    'language': ['en'] * 10000,
    'publish_date': ['2023-01-01'] * 10000,
    'progression': [0.5] * 10000,
    'media': ['image.jpg'] * 10000,
    'type': ['article'] * 10000
}
df_dummy = pd.DataFrame(dummy_data)
df_dummy.to_csv(path, index=False)

pdf = pd.read_csv(path)

# Identifiant unique
pdf["id"] = range(len(pdf))

# Exploration
display(pdf)
print(f"Dimensions de pdf: {pdf.shape}")
print(f"Types de colonnes de pdf:\n{pdf.dtypes}")
print(f"Valeurs manquantes par colonne dans pdf:\n{pdf.isnull().sum()}")

# Sous-ensemble de développement (1000 premières lignes)
pdf_subset = pdf.head(1000).reset_index(drop=True)
print(f"Dimensions de pdf_subset: {pdf_subset.shape}")

,title,topic,url,domain,language,publish_date,progression,media,type,id
0,Dummy News Title 0,Topic 0,http://example.com/0,example.com,en,2023-01-01,0.5,image.jpg,article,0
1,Dummy News Title 1,Topic 1,http://example.com/1,example.com,en,2023-01-01,0.5,image.jpg,article,1
2,Dummy News Title 2,Topic 2,http://example.com/2,example.com,en,2023-01-01,0.5,image.jpg,article,2
3,Dummy News Title 3,Topic 3,http://example.com/3,example.com,en,2023-01-01,0.5,image.jpg,article,3
4,Dummy News Title 4,Topic 4,http://example.com/4,example.com,en,2023-01-01,0.5,image.jpg,article,4
...,...,...,...,...,...,...,...,...,...,...
9995,Dummy News Title 9995,Topic 0,http://example.com/9995,example.com,en,2023-01-01,0.5,image.jpg,article,9995
9996,Dummy News Title 9996,Topic 1,http://example.com/9996,example.com,en,2023-01-01,0.5,image.jpg,article,9996
9997,Dummy News Title 9997,Topic 2,http://example.com/9997,example.com,en,2023-01-01,0.5,image.jpg,article,9997
9998,Dummy News Title 9998,Topic 3,http://example.com/9998,example.com,en,2023-01-01,0.5,image.jpg,article,9998


Dimensions de pdf: (10000, 10)
Types de colonnes de pdf:
title            object
topic            object
url              object
domain           object
language         object
publish_date     object
progression     float64
media            object
type             object
id                int64
dtype: object
Valeurs manquantes par colonne dans pdf:
title           0
topic           0
url             0
domain          0
language        0
publish_date    0
progression     0
media           0
type            0
id              0
dtype: int64
Dimensions de pdf_subset: (1000, 10)


Exercice 2 : Vectorisation avec des transformateurs de phrases


Dans cet exercice, nous allons transformer nos données textuelles (titres d'articles) en représentations numériques appelées plongements lexicaux . Cette étape est cruciale pour permettre aux machines de comprendre et de traiter des données textuelles dans des tâches telles que la recherche de similarités, le clustering et l'apprentissage automatique. Nous utiliserons Sentence Transformers , une bibliothèque populaire pour générer des représentations vectorielles denses de texte.



Pourquoi cette étape est importante :

Les machines ne peuvent pas traiter directement du texte brut ; elles ont besoin d’entrées numériques. Les vecteurs d’intégration (embeddings) sont des vecteurs denses qui capturent le sens et le contexte du texte. En générant des vecteurs d’intégration pour nos titres d’articles, nous les rendons utilisables pour des tâches en aval telles que la recherche de similarités ou l’alimentation de modèles d’apprentissage automatique.



In [26]:
from sentence_transformers import SentenceTransformer, InputExample

# pdf_subset est déjà défini à l'exercice 1

In [27]:
def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Helper function that outputs a sentence_transformer guid, label, and text.
    """
    return InputExample(texts=[doc1])  # Le titre est passé comme texte d'entrée

In [28]:
faiss_train_examples = pdf_subset.apply(
    lambda x: example_create_fn(str(x["title"])), axis=1
).tolist()

faiss_train_examples[:10]  # Vérification des 10 premiers

In [29]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
# Modèle léger (~80MB), 384 dimensions, excellent rapport qualité/vitesse

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [30]:
titles_list = pdf_subset["title"].astype(str).tolist()
print(f"Nombre de valeurs: {len(titles_list)}")
print(f"Exemple : {titles_list[0]}")

Nombre de valeurs: 1000
Exemple : Dummy News Title 0


In [31]:
faiss_title_embedding = model.encode(
    titles_list,
    show_progress_bar=True  # Optionnel mais utile pour suivre la progression
)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [32]:
len(faiss_title_embedding), len(faiss_title_embedding[0])
# Attendu → (1000, 384)
# 1000 embeddings (un par titre), chaque vecteur de dimension 384

(1000, 384)

Exercice 3 : Indexation et recherche FAISS


Dans cet exercice, nous utiliserons FAISS (Facebook AI Similarity Search) pour créer un index des plongements lexicaux générés lors de l'exercice précédent. Cela nous permettra d'effectuer des recherches de similarité rapides et efficaces sur de vastes ensembles de vecteurs. L'objectif est de permettre la récupération des articles d'actualité les plus pertinents en fonction de la requête d'un utilisateur.



Pourquoi cette étape est importante :


FAISS est une bibliothèque conçue pour effectuer des recherches de similarité à grande échelle. La recherche efficace au sein d'embeddings (vecteurs de grande dimension) devient complexe lorsqu'on travaille avec ces derniers. FAISS propose des algorithmes optimisés d'indexation et de recherche, permettant de retrouver des éléments similaires en quelques millisecondes, même dans de vastes ensembles de données.

In [33]:
pdf_to_index = pdf_subset.copy()                          # Le sous-ensemble de l'exercice 1
id_index = np.array(pdf_to_index["id"].tolist())          # Tableau d'IDs uniques (int64 requis par FAISS)

In [34]:
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
# IndexFlatIP  → produit scalaire (= cosinus sur vecteurs normalisés)
# IndexIDMap   → associe chaque vecteur à son ID original

# Normalisation des embeddings pour la similarité cosinus
faiss.normalize_L2(faiss_title_embedding)

index_content.add_with_ids(faiss_title_embedding, id_index)

print(f"Vecteurs indexés : {index_content.ntotal}")  # → 1000

Vecteurs indexés : 1000


In [35]:
def search_content(query, pdf_to_index, k=3):
    # Encoder la requête
    query_vector = model.encode([query]).astype("float32")
    faiss.normalize_L2(query_vector)                     # Normaliser la requête

    # Recherche dans l'index
    top_k = index_content.search(query_vector, k)        # Retourne (distances, ids)
    ids = top_k[1][0]                                    # IDs des k voisins les plus proches
    similarities = top_k[0][0]                           # Scores de similarité correspondants

    # Récupérer les articles depuis le DataFrame
    results = pdf_to_index[pdf_to_index["id"].isin(ids)].copy()
    results["similarities"] = similarities               # Ajouter les scores

    return results

In [36]:
display(search_content("animal", pdf_to_index, k=5))

,title,topic,url,domain,language,publish_date,progression,media,type,id,similarities
177,Dummy News Title 177,Topic 2,http://example.com/177,example.com,en,2023-01-01,0.5,image.jpg,article,177,0.243359
178,Dummy News Title 178,Topic 3,http://example.com/178,example.com,en,2023-01-01,0.5,image.jpg,article,178,0.240535
247,Dummy News Title 247,Topic 2,http://example.com/247,example.com,en,2023-01-01,0.5,image.jpg,article,247,0.239540
266,Dummy News Title 266,Topic 1,http://example.com/266,example.com,en,2023-01-01,0.5,image.jpg,article,266,0.236886
274,Dummy News Title 274,Topic 4,http://example.com/274,example.com,en,2023-01-01,0.5,image.jpg,article,274,0.236105


Exercice 4 : Collection et requêtes ChromaDB


Dans cet exercice, nous présenterons ChromaDB , une base de données vectorielle open source conçue pour stocker, indexer et interroger des vecteurs d'embeddings. ChromaDB simplifie la manipulation des embeddings et, contrairement à FAISS, gère automatiquement la tokenisation, l'embedding et l'indexation sans nécessiter de génération manuelle d'embeddings. Elle est donc idéale pour l'intégration avec des applications basées sur des modèles de langage étendus (LLM ), notamment pour la création de systèmes de questions-réponses ou de moteurs de recherche.



Pourquoi cette étape est importante :

Une fois les représentations vectorielles de nos données générées, l'étape suivante consiste à les stocker et à les interroger efficacement. ChromaDB offre une interface de haut niveau pour la gestion de ces représentations et prend en charge les métadonnées, ce qui en fait une solution idéale pour développer des applications telles que la recherche documentaire ou les systèmes de questions-réponses. À l'aide de ChromaDB, nous montrons comment intégrer ces représentations vectorielles dans un flux de travail concret permettant d'interroger et de récupérer des documents pertinents.



Instructions:

In [37]:
chroma_client = chromadb.Client()
collection_name = "my_news"

# Supprimer la collection si elle existe déjà
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

Creating collection: 'my_news'


In [38]:
display(pdf_subset)

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[str(i) for i in pdf_subset["id"][:100].tolist()]  # IDs uniques en string (requis par ChromaDB)
)

print(f"Documents ajoutés : {collection.count()}")  # → 100

,title,topic,url,domain,language,publish_date,progression,media,type,id
0,Dummy News Title 0,Topic 0,http://example.com/0,example.com,en,2023-01-01,0.5,image.jpg,article,0
1,Dummy News Title 1,Topic 1,http://example.com/1,example.com,en,2023-01-01,0.5,image.jpg,article,1
2,Dummy News Title 2,Topic 2,http://example.com/2,example.com,en,2023-01-01,0.5,image.jpg,article,2
3,Dummy News Title 3,Topic 3,http://example.com/3,example.com,en,2023-01-01,0.5,image.jpg,article,3
4,Dummy News Title 4,Topic 4,http://example.com/4,example.com,en,2023-01-01,0.5,image.jpg,article,4
...,...,...,...,...,...,...,...,...,...,...
995,Dummy News Title 995,Topic 0,http://example.com/995,example.com,en,2023-01-01,0.5,image.jpg,article,995
996,Dummy News Title 996,Topic 1,http://example.com/996,example.com,en,2023-01-01,0.5,image.jpg,article,996
997,Dummy News Title 997,Topic 2,http://example.com/997,example.com,en,2023-01-01,0.5,image.jpg,article,997
998,Dummy News Title 998,Topic 3,http://example.com/998,example.com,en,2023-01-01,0.5,image.jpg,article,998


Documents ajoutés : 100


In [39]:
results = collection.query(
    query_texts=["space"],   # ChromaDB génère l'embedding automatiquement
    n_results=10             # Les 10 documents les plus proches sémantiquement
)

print(json.dumps(results, indent=4))

{
    "ids": [
        [
            "13",
            "15",
            "3",
            "17",
            "38",
            "2",
            "5",
            "45",
            "27",
            "94"
        ]
    ],
    "embeddings": null,
    "documents": [
        [
            "Dummy News Title 13",
            "Dummy News Title 15",
            "Dummy News Title 3",
            "Dummy News Title 17",
            "Dummy News Title 38",
            "Dummy News Title 2",
            "Dummy News Title 5",
            "Dummy News Title 45",
            "Dummy News Title 27",
            "Dummy News Title 94"
        ]
    ],
    "uris": null,
    "included": [
        "metadatas",
        "documents",
        "distances"
    ],
    "data": null,
    "metadatas": [
        [
            {
                "topic": "Topic 3"
            },
            {
                "topic": "Topic 0"
            },
            {
                "topic": "Topic 3"
            },
            {
        

Exercice 5 : Réponse aux questions avec un modèle de visage faisant un câlin


Dans cet exercice, nous allons mettre en pratique tous les éléments en construisant un système de questions-réponses (Q/R) utilisant le modèle de langage Hugging Face . En combinant la recherche documentaire (via ChromaDB) avec la génération de texte (via Hugging Face), nous créons un pipeline simple mais puissant où un modèle génère des réponses en fonction du contexte pertinent.



Pourquoi cette étape est importante :


La récupération des documents pertinents ne représente que la moitié du travail. L'étape suivante consiste à générer des réponses pertinentes à partir de ces contenus. Il s'agit d'une technique fondamentale des systèmes modernes de génération augmentée par la récupération (RAG) , où un modèle de langage exploite à la fois des connaissances pré-entraînées et des informations externes pour répondre aux questions avec plus de précision. En intégrant ChromaDB et les transformateurs Hugging Face , nous simulons un processus de questions-réponses réaliste.



In [40]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Modèle
model_id  = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model  = AutoModelForCausalLM.from_pretrained(model_id)

# Pipeline
pipe = pipeline("text-generation", model=lm_model,
                tokenizer=tokenizer, max_new_tokens=512, device_map="auto")

# Prompt
question        = "What's the latest news on space development?"
context         = " ".join([f"#{str(i)}" for i in results["documents"][0]])
prompt_template = f"Relevant context: {context}\n\nThe user's question: {question}"

# Génération
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=512) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Relevant context: #Dummy News Title 13 #Dummy News Title 15 #Dummy News Title 3 #Dummy News Title 17 #Dummy News Title 38 #Dummy News Title 2 #Dummy News Title 5 #Dummy News Title 45 #Dummy News Title 27 #Dummy News Title 94

The user's question: What's the latest news on space development?

You can enter a topic for the past or present of the question in the form of a question mark. You can also enter a question mark (or any other form of question mark) to enter a question from any other topic.

You can also enter a question mark (or any other form of question mark) to enter a question from any other topic. Question name: Your name.

Your name. Question title: Your title.

Your title. Question number: The number of questions you have entered.

The number of questions you have entered. Question format: The format of the question.

The format of the question. Question type: The type of question.

The type of question. Question time: The time after which the question is answered.

The ti